# Arabic Wav2Vec2 – Low-Latency WebSocket Streaming (Ziel: ~400ms)

**Modell:** `jonatasgrosman/wav2vec2-large-xlsr-53-arabic`

## Was dieses Notebook tut

1. Lädt das Wav2Vec2-Arabisch-Modell in **fp16** auf die Colab-GPU (ca. 2x schneller als fp32).
2. Startet einen **WebSocket-Server**, der eingehendes Mikrofon-Audio in Echtzeit puffert, mit Voice-Activity-Detection (VAD) in Sprachsegmente zerlegt und laufend Teil- und Endtranskripte zurückschickt.
3. Macht den Server über einen **ngrok-Tunnel** öffentlich erreichbar (Colab hat keinen offenen Port).
4. Erzeugt eine **`client.html`**, die du lokal im Browser öffnest: Mikrofon aufnehmen, Audio an den Server streamen, Transkript live anzeigen – inklusive einer **Live-Latenzanzeige** (Millisekunden von "Chunk gesendet" bis "Antwort erhalten"), damit du selbst siehst, ob du die 400ms triffst.

## Ehrliche Einordnung zum 400ms-Ziel

wav2vec2 ist architektonisch **kein** kausales Streaming-Modell (volle, nicht-kausale Attention/Convolutions). Mit den Optimierungen hier (fp16, kleine Chunks, kurze VAD-Fenster) kommst du im **Best Case** (schnelle Internetverbindung, T4/L4-GPU, kurze Sprachsegmente) in die Nähe von 400ms für Teilergebnisse. Realistisch schwankt es je nach:
- **Netzwerk:** Du bist in Deutschland, der ngrok-Tunnel und der Colab-Host können aber in den USA liegen → das allein kann schon 50–200ms Rundlaufzeit kosten, die du nicht wegoptimieren kannst, ohne selbst zu hosten.
- **GPU-Auslastung:** Colab-GPUs werden mit anderen Nutzern geteilt (Free-Tier), Inferenzzeit kann schwanken.
- **Chunk-/Segmentlänge:** kürzere Chunks = niedrigere Latenz, aber schlechtere Erkennungsqualität (weniger akustischer Kontext).

Die Latenzanzeige im HTML-Client macht das messbar, statt es zu erraten. Wenn du nach dem Testen konstant über 400ms liegst, sind die nächsten Hebel (nicht in diesem Notebook enthalten): eigener EU-Server statt Colab+ngrok, ONNX Runtime/TensorRT-Export des Modells, oder Wechsel auf ein echtes Streaming-Modell wie NVIDIA Nemotron 3.5 ASR (siehe unser vorheriges Gespräch).

## Setup

1. Laufzeit → Laufzeittyp ändern → **T4 GPU** (oder besser, falls verfügbar)
2. Zellen der Reihe nach ausführen
3. Eigenen ngrok-Authtoken eintragen: https://dashboard.ngrok.com/get-started/your-authtoken
4. Ausgegebene `wss://...`-URL in `client.html` eintragen, verbinden, sprechen


In [ ]:
!pip install -q transformers torch soundfile websockets pyngrok webrtcvad nest_asyncio numpy

In [ ]:
import torch, time
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import numpy as np

MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = device == "cuda"
print("Device:", device, "| fp16:", USE_FP16)
if device == "cpu":
    print("WARNUNG: Keine GPU aktiv -> 400ms-Ziel ist so nicht erreichbar. Laufzeittyp auf GPU stellen!")

processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device)
if USE_FP16:
    model = model.half()
model.eval()

SAMPLE_RATE = 16000

@torch.inference_mode()
def transcribe(audio_np: np.ndarray):
    """Gibt (text, inferenz_ms) zurueck."""
    if audio_np.size == 0:
        return "", 0.0
    t0 = time.perf_counter()
    inputs = processor(audio_np, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    if USE_FP16:
        input_values = input_values.half()
    logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    text = processor.batch_decode(pred_ids)[0]
    ms = (time.perf_counter() - t0) * 1000
    return text, ms

# Warmup: erster Forward-Pass ist immer langsamer (CUDA-Kernel-Kompilierung)
_ = transcribe(np.zeros(SAMPLE_RATE // 2, dtype=np.float32))
print("Modell geladen und aufgewaermt.")


## Chunking-Logik (auf niedrige Latenz getrimmt)

Kürzere Werte als in einem Standard-Setup, um näher an 400ms zu kommen:
- `partial_interval_ms=280` — alle ~280ms neues Audio gibt es ein Teilergebnis
- `silence_ms_to_finalize=250` — nach 250ms Stille wird das Segment finalisiert
- `frame_ms=20` — feinere VAD-Auflösung als Standard (30ms)

Jedes Event trägt `inference_ms` (reine Modell-Rechenzeit) mit, getrennt von der Netzwerklaufzeit.


In [ ]:
import webrtcvad

class StreamingSession:
    def __init__(self, sample_rate=16000, frame_ms=20, vad_aggressiveness=2,
                 silence_ms_to_finalize=250, partial_interval_ms=280):
        self.sample_rate = sample_rate
        self.frame_ms = frame_ms
        self.frame_bytes = int(sample_rate * frame_ms / 1000) * 2  # 16-bit PCM mono
        self.vad = webrtcvad.Vad(vad_aggressiveness)
        self.buffer = bytearray()
        self.speech_audio = bytearray()
        self.silence_ms = 0
        self.silence_ms_to_finalize = silence_ms_to_finalize
        self.last_partial_len = 0
        self.partial_interval_bytes = int(sample_rate * partial_interval_ms / 1000) * 2

    def _pcm_to_np(self, pcm_bytes):
        return np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0

    def add_audio(self, pcm_bytes):
        events = []
        self.buffer.extend(pcm_bytes)

        while len(self.buffer) >= self.frame_bytes:
            frame = bytes(self.buffer[:self.frame_bytes])
            del self.buffer[:self.frame_bytes]

            is_speech = self.vad.is_speech(frame, self.sample_rate)
            if is_speech:
                self.speech_audio.extend(frame)
                self.silence_ms = 0
                if len(self.speech_audio) - self.last_partial_len >= self.partial_interval_bytes:
                    text, ms = transcribe(self._pcm_to_np(bytes(self.speech_audio)))
                    events.append(("partial", text, ms))
                    self.last_partial_len = len(self.speech_audio)
            else:
                if len(self.speech_audio) > 0:
                    self.silence_ms += self.frame_ms
                    if self.silence_ms >= self.silence_ms_to_finalize:
                        text, ms = transcribe(self._pcm_to_np(bytes(self.speech_audio)))
                        events.append(("final", text, ms))
                        self.speech_audio = bytearray()
                        self.last_partial_len = 0
                        self.silence_ms = 0
        return events


## WebSocket-Server + ngrok-Tunnel

Dein ngrok-Authtoken ist bereits in der Zelle unten eingetragen. Zelle ausführen -> die ausgegebene `wss://...`-URL kopieren und in `client.html` (Feld ganz oben) einfügen.


In [ ]:
import asyncio, json
import websockets
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

NGROK_AUTHTOKEN = "3HtsWbCZvpQuQVAYdx8yTMVVdAx_bAd2nWnDJcX5FeBgyJm1"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

PORT = 8765

async def handler(websocket):
    session = StreamingSession()
    print("Client verbunden")
    try:
        async for message in websocket:
            if isinstance(message, (bytes, bytearray)):
                for kind, text, ms in session.add_audio(message):
                    await websocket.send(json.dumps({
                        "type": kind,
                        "text": text,
                        "inference_ms": round(ms, 1),
                    }))
    except websockets.exceptions.ConnectionClosed:
        print("Client getrennt")

async def run_server():
    async with websockets.serve(handler, "0.0.0.0", PORT, max_size=None):
        await asyncio.Future()

http_tunnel = ngrok.connect(PORT, "http")
public_url = http_tunnel.public_url
ws_url = public_url.replace("https://", "wss://").replace("http://", "ws://")
print("Oeffentliche WebSocket-URL fuer client.html:", ws_url)

asyncio.run(run_server())


## Test-Client (Browser, Mikrofon, Live-Latenzanzeige)

`client.html` lokal herunterladen und im Browser öffnen (Chrome empfohlen). Die `wss://...`-URL von oben eintragen, verbinden, Mikrofonzugriff erlauben. Die Anzeige "Latenz" zeigt die Zeit von "Chunk gesendet" bis "Antwort erhalten" (Netzwerk + Inferenz zusammen) sowie separat die reine Modell-Inferenzzeit vom Server.


## Wenn 400ms nicht konstant erreicht werden

Reihenfolge der Hebel, grob nach Aufwand:

1. `partial_interval_ms` / `silence_ms_to_finalize` weiter senken (Qualität sinkt)
2. `vad_aggressiveness` anpassen (0=locker...3=streng) — beeinflusst, wie schnell Sprache/Stille erkannt wird
3. Colab Pro/Pro+ für zuverlässigere, schnellere GPU statt Free-Tier-T4
4. Modell als ONNX exportieren + ONNX Runtime GPU (oft spuerbar schneller als natives PyTorch)
5. Statt Colab+ngrok: eigener Server in einem EU-Rechenzentrum (z.B. Hetzner, OVH) — eliminiert einen Grossteil der Netzwerklatenz von Deutschland aus
6. Architektur wechseln: ein echtes Streaming-Modell wie NVIDIA Nemotron 3.5 ASR (siehe vorheriges Gespraech) statt wav2vec2 — das ist der einzige Weg, strukturell (nicht nur durch Tuning) niedrigere Latenz zu bekommen
